# Semana 1 — Fundamentos da DataFrame API

Estudo dos verbos básicos de transformação em PySpark: `select`, `filter`, `withColumn` e `when/otherwise`, com dados sintéticos de pedidos.

Parte do plano de reposicionamento para engenharia de dados.


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
import random
from datetime import date, timedelta

random.seed(42)
produtos = ["VR", "VA", "VT", "AUXILIO"]
base = date(2026, 1, 1)

dados = [
    (
        i,
        random.randint(1000, 1050),
        random.choice(produtos),
        base + timedelta(days=random.randint(0, 300)),
        round(random.uniform(50, 5000), 2),
        round(random.uniform(0, 0.15), 4),
    )
    for i in range(1, 50001)
]

pedidos = spark.createDataFrame(
    dados,
    ["id_pedido", "id_cliente", "produto", "data_pedido", "valor_bruto", "pct_desconto"]
)
pedidos.show(5)
pedidos.printSchema()


In [0]:
pedidos.select("id_pedido", "produto", "valor_bruto").show(5)

In [0]:
pedidos.filter((F.col("produto") == "VR") & (F.col("valor_bruto") > 1000)).show(5)

In [0]:
pedidos.withColumn("valor_liquido", F.col("valor_bruto") * (1 - F.col("pct_desconto"))).show(5)

In [0]:
pedidos.show(5)

In [0]:
pedidos_com_liquido = pedidos.withColumn("valor_liquido", F.col("valor_bruto") * (1 - F.col("pct_desconto")))
pedidos_com_liquido.show(5)

In [0]:
pedidos.select("id_pedido", "produto", "valor_bruto").filter((F.col("produto") == "VR") & (F.col("valor_bruto") > 2000.00)).show(5) 

In [0]:
pedidos.withColumn("valor_desconto", F.col("valor_bruto") * (F.col("pct_desconto"))).show(5)

In [0]:
pedidos.withColumn("categoria",F.when(F.col("valor_bruto") > 3000, "alto").when(F.col("valor_bruto") > 1000, "medio").otherwise("baixo")).show(5)

# Semana 1 — Fundamentos da DataFrame API

Estudo dos verbos básicos de transformação em PySpark: `select`, `filter`, `withColumn`, `when/otherwise`, `schema`, `escrita em Delta`, com dados sintéticos de pedidos.

Parte do plano de reposicionamento para engenharia de dados.

In [0]:
schema_pedidos = StructType([
    StructField("id_pedido",    IntegerType(),      False),
    StructField("id_cliente",   IntegerType(),      False),
    StructField("produto",      StringType(),       True),
    StructField("data_pedido",  DateType(),         True),
    StructField("valor_bruto",  DecimalType(12, 2), True),
    StructField("pct_desconto", DecimalType(6, 4),  True),
])

In [0]:
from decimal import Decimal

random.seed(42)
dados = [
    (
        i,
        random.randint(1000, 1050),
        random.choice(produtos),
        base + timedelta(days=random.randint(0, 300)),
        Decimal(str(round(random.uniform(50, 5000), 2))),
        Decimal(str(round(random.uniform(0, 0.15), 4))),
    )
    for i in range(1, 50001)
]

pedidos = spark.createDataFrame(dados, schema_pedidos)
pedidos.printSchema()
pedidos.show(5)

In [0]:
pedidos_com_liquido = pedidos.withColumn("valor_liquido", F.col("valor_bruto") * (1 - F.col("pct_desconto")))
pedidos_com_liquido.show(5)

In [0]:
pedidos.withColumn("valor_desconto", F.col("valor_bruto") * (F.col("pct_desconto"))).show(5)


In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")

In [0]:
pedidos.write.format("delta").mode("overwrite").saveAsTable("bronze.pedidos")

In [0]:
pedidos_lido = spark.read.table("bronze.pedidos")
pedidos_lido.printSchema()
pedidos_lido.show(5)